# **Accidents en Belgique** : Commentaires

---

## Introduction

Le premier objectif de ce projet est d'explorer les possibles causes des accidents de la voirie impliquant des lésions corporelles (légères ou graves). Bien entendu certaines semblent déjà évidentes, les conditions météo par exemple, ou le densité du réseau routier. Dans ce projet, je chercherai aussi des corrélations plus exotiques. Les communes au revenu administratif median plus élevé observent-elles plus ou moins d'accidents ? L'occupation des sols à-t-elle un impact sur la quantité d'accident ? Le statut civil ou encore le genre prédominant dans les villes influence-t-il la fréquence des accidents ?   

Le deuxième objectif sera de déterminer les endroits les plus impactés, quelles autoroutes sont les plus meurtrières ? Les ambulances sont-elles réparties de manière optimale ? Recouvrent-elles les zones les plus dangereuses ? 

---

## Raw to Bronze

### Objectifs : 
Rendre lisibles les données récupérées. 

### Provenances des données :

L'ensemble des données est composé de 6 thèmes pour un total de 14 fichiers.

- **Accidents** :    
    Fichier excel principal contenant les accidents répertoriés entre 2017 et 2024.   
    lien : https://statbel.fgov.be/fr/open-data/geolocalisation-des-accidents-de-la-circulation-2017-2024

- **Population** :    
    Ensemble de 8 fichiers excel comptant le nombre d'habitant par commune, genre, etat civil et par age sur une année.  
    lien (2024) : https://statbel.fgov.be/fr/open-data/population-par-lieu-de-residence-nationalite-etat-civil-age-et-sexe-15 

- **Réseau routier** :   
    Ensemble de 2 fichiers (composés de plusieurs "*sous-fichiers*"). Le premier listant les autoroutes belges avec leur tracé. Le second listant les routes belges et leur tracé. Ces fichiers contiennent des *geometry* comme valeurs, à utiliser avec geopandas.   
    lien : https://www.geo.be/catalog/details/a9187022-0bf6-4e96-a787-25476d9f1173?l=fr

- **Occupation du sol** :   
    Fichier excel donnant le nbr d'hectare par occupation du sol, par commune.   
    lien : https://statbel.fgov.be/fr/themes/construction-logement/utilisation-du-sol#figures  

- **Revenus** :     
    Fichier excel contenant 7 feuilles (intéressantes pour mon projet) donnant notamment la medianne du revenu et le risque de pauvreté administratif par commune.   
    Remarques : 
     - En raison de la faible population de Herstappe les résultats de Herstappe et de Tongres ont été combinés.
     - Il n'est peut-être pas pertinent de comparer les revenus avant et après 2020, suite à une amélioration de la méthode   

    lien : https://statbel.fgov.be/fr/themes/datalab/revenu-disponible-administratif

- **Soins** :   
    Fichier excel répertoriant l'ensemble des points de départ (longitude, latitude) des ambulances.   
    lien : https://www.health.belgium.be/sites/default/files/media/files/2026-04/ambulances_01042026_fr.xlsx

---

## Bronze to Silver

### Objectifs :

Rendre les données utilisables et pertinentes

### Structure

La partie **Silver** se construit en 5 scripts.

- *accidents.py* : nettoie les données brutes sur les accidents et les enrichi en ajoutant les colonnes du nom de l'autoroute si c'est le cas, la largeur de la route, ainsi que la densité du réseau routier à proximité. 

- *population.py* : à nécessité une modification des valeurs de départ. En effet, le code REFNIS de certaines communes à changé en 2019, suite à des fusions. J'ai donc fait le choix de modifier le code REFNIS de 2017 et 2018 en celui de 2024 pour uniformiser le tout.

- *occupation_sol.py* : procède à un nettoyage, dont une lourde sélection de ligne et de colonne. Dépivot les années pour pivoter ensuite le détail de l'occupation des sols. 

- *revenus.py* : simple nettoyage. 

- ***bronze_to_silver.py*** : lance les fonctions implémentées dans les précédents scripts, une de chargement et une deuxième de nettoyage / enrichissement.

### Analyse qualitative des données

Le notebook *analyse_qualitative.ipynb* fournit des informations concernant le jeu de donnée, comme la taille du data set, les typages, quelques valeurs descriptives pour les variables numériques, etc.

---

## Silver to Gold

### Objectifs :

Réaliser une modélisation dimensionnelle.

### Modélisation dimensionnelle

Le **Gold** est organisé selon un *schéma en étoile* composé d'une table de faits et de sept dimensions.

#### Dimensions

| Dimension        |                        Description                                                           |
| ---------------- | --------------------------------------------------------------------- |
| `dim_geo`        | Entité administrative géographique (commune, province, région, code REFNIS de la commune).     |
| `dim_date`       | Dimension temporelle.       |
| `dim_condition`  | Conditions indépendantes de l'accident.             |
| `dim_situation`  | Relatif à l'évènement. |
| `dim_population` | Information sur le type de population (genre, état civil, age) par commune.                     |
| `dim_occupation` | Occupation du sol par commune.                                      |
| `dim_revenus`    | Revenus de la commune par commune.                          |

#### Détails des dimensions
<div style="display:flex; flex-wrap:wrap; gap:20px; align-items:flex-start;">
<div>

| **dim_geo**             |
| -------------------- |
| `ID_geo`             |
| `GEO_commune_REFNIS` |
| `GEO_commune`        |
| `GEO_province`       |
| `GEO_region`         |

</div>
<div>

| **dim_date**      |
| ------------- |
| `ID_date`     |
| `DT_annee`    |
| `DT_mois`     |
| `DT_nom_mois` |
| `DT_heure`    |
| `DT_date`     |

</div>
<div>

| **dim_condition**                  |
| ------------------------- |
| `ID_condition`            |
| `COND_carrefour`          |
| `COND_meteo`              |
| `COND_route`              |
| `COND_agglomeration`      |
| `COND_lumiere`            |
| `COND_type_route`         |
| `COND_obstacles`          |
| `COND_routes_largeur_cat` |
| `COND_densite`            |

</div>
<div>

| **dim_situation**             |
| -------------------- |
| `ID_situation`       |
| `ACC_etat_victimes`  |
| `ACC_vehicule_1`     |
| `ACC_vehicule_2`     |
| `ACC_collision_type` |
| `ACC_nom_autoroute`  |

</div>
<div>

| **dim_population**          |
| ----------------- |
| `ID_population`   |
| `ID_geo`          |
| `ID_date`         |
| `P_age_moyen`     |
| `P_age_median`    |
| `P_nbr_habitants` |
| `P_%_F`           |
| `P_%_M`           |
| `P_%_Célibataire` |
| `P_%_Divorcé`     |
| `P_%_Marié`       |
| `P_%_Veuf`        |

</div>
<div>

| **dim_occupation**                                         |
| ------------------------------------------------ |
| `ID_occupation`                                  |
| `ID_geo`                                         |
| `ID_date`                                        |
| *(ensemble des indicateurs d'occupation du sol)* |

</div>
<div>

| **dim_revenus**                   |
| -------------------------- |
| `ID_revenus`               |
| `ID_geo`                   |
| `ID_date`                  |
| `REV_pourcentage_manquant` |
| `REV_median`               |
| `REV_risque_pauvrete`      |

</div>
<div>

| **Table de faits**   | Référence        |
| --------------- | ---------------- |
| `GEO_longitude`    |  -    |
| `GEO_latitude`    |  -   |
| `SOIN_min_distance`    | -    |
| `ID_geo`        | `dim_geo`        |
| `ID_date`       | `dim_date`       |
| `ID_condition`  | `dim_condition`  |
| `ID_situation`  | `dim_situation`  |
| `ID_population` | `dim_population` |
| `ID_occupation` | `dim_occupation` |
| `ID_revenus`    | `dim_revenus`    |

